<a href="https://colab.research.google.com/github/Stronghold1388/Agent_experiments/blob/main/Personal_chef(Lanfchain_practice).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain langchain-openai huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.16
    Uninstalling langchain-protocol-0.0.16:
      Successfully uninstalled langchain-protocol-0.0.16
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
  Attempting uninst

In [2]:
#model_invocation
from huggingface_hub.inference._providers import openai
from langchain_openai.chat_models import ChatOpenAI
from huggingface_hub import InferenceClient
from google.colab import userdata
import os

client=InferenceClient(model='openai/gpt-oss-120b')
model=ChatOpenAI(
    openai_api_key=userdata.get('HF_TOKEN'),
    model_name='openai/gpt-oss-120b',
    base_url="https://router.huggingface.co/v1",
    max_tokens=300,
    temperature=0.7,
    streaming=True
)

In [3]:
!pip install tavily-python
from langchain.tools import tool
from tavily import TavilyClient
from typing import Dict,Any
from google.colab import userdata
import os

os.environ['TAVILY_API_KEY']=userdata.get('TAVILY_API_KEY')
tavily_client=TavilyClient()

#tools
@tool ('search_web',description='search the web for information')
def ws_tool(query,str):
  return tavily_client.search(query,max_results=3)

In [4]:

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

system_prompt="""You are a professional chef's assistant. You can suggest and improve recipe according to -occasion,deitary restrictions, demands, taste.
               You are able to use the given tools to reach best possible recipe for the user.
               you will always provide recipe in bullet points.
               you must suggest drinks that goes well with the suggested food.
               Always follow this output format=
               Recipe: (example-  Chicken curry
               -Link-link of the recipe
               -500g fresh chiken
               -ginger
               -garlic
               -100g onion ..etc)
               Drinks: (only name)
               Always follow this process-Think>Action>Observation(this process can repeat n times, till a satisfactory result is not achieved)"""

chef_agent=create_agent(model=model,
                        tools=[ws_tool],
                        system_prompt=system_prompt,
                        checkpointer=InMemorySaver() )

question=HumanMessage(content="Tommorow is my wife's bithday , she likes chicken dishes with rice.Can you suggest me some recipe for her?")
config={'configurable':{'thread_id':'1'}}

result=chef_agent.invoke(
                         {"messages":[question]},
                          config=config
                         )
print(result['messages'][-1].content)



**Recipe:** Chicken Biryani (celebratory Indian rice‑and‑chicken dish)  
- Link: https://www.andy-cooks.com/blogs/recipes/chicken-biryadi  
- 1 kg bone‑in chicken thighs, skin removed, cut into large pieces  
- 500 g basmati rice, rinsed and soaked 30 min  
- 2 large onions, thinly sliced  
- 3 tbsp plain yogurt  
- 2 tbsp ginger‑garlic paste  
- 2 tbsp biryani masala (store‑bought


In [5]:
question=HumanMessage(content='can you make it light .')
config= {'configurable':{'thread_id':'1'}}

result=chef_agent.invoke(
                         {"messages":[question]},
                          config=config
                         )
print(result['messages'][-1].content)


**Recipe:** Light Chicken Biryani  
- Link: https://www.storyofcooks.com/light-chicken-biryani  
- 500 g skinless chicken breast, cut into bite‑size pieces (or 400 g boneless chicken thighs for extra juiciness)  
- 1 ½ cups brown basmati rice, rinsed and soaked 20 min (or 1 ½ cups cauliflower‑rice for an ultra‑light option)  
- 1 large onion, finely chopped (½ for the paste,½ for garnish)  
- 2 tbsp low‑fat Greek yogurt (adds creaminess without heavy cream)  
- 1 tbsp ginger‑garlic paste (equal parts fresh ginger and garlic, blended)  
- 1  tsp ground turmeric  
- 1  tsp garam masala (store‑bought or homemade)  
- ½  tsp ground cumin  



In [6]:
question=HumanMessage(content='what about dessert.')
config= {'configurable':{'thread_id':'1'}}
result=chef_agent.invoke(
                         {"messages":[question]},
                          config=config
                         )
print(result['messages'][-1].content)

**Recipe:** Light Mango Yogurt Parfait  
- Link: https://www.healthylittlebites.com/mango-yogurt-parfait  
- 2  cups low‑fat Greek yogurt (plain)  
- 1  cup fresh mango puree (blend ripe mangoes with a splash of lime juice)  
- 2  tbsp honey or maple syrup (optional, to taste)  
- ½  tsp cardamom powder (adds a subtle warm spice)  
- ¼  cup toasted unsalted coconut flakes (for a light


In [7]:
question=HumanMessage(content="don't you think with cake and all it would be toomuch sweet")

result=chef_agent.invoke(
                         {"messages":[question]},
                          config=config
                         )
print(result['messages'][-1].content)
#

**Recipe:** Light Citrus‑Honey Yogurt Parfait  
- Link: https://www.healthylittlebites.com/mango-yogurt-parfait (adapted for citrus)  
- 2  cups low‑fat plain Greek yogurt  
- 1  cup fresh orange (or tangerine) segments, cut into bite‑size pieces  
- ½  cup fresh pineapple chunks (optional
